# bf16@100k vs fp32 — both reason, both fail OOD by **over-generating**

> **Correction up front.** An earlier single sample made it look like the bf16 100k model emits
> degenerate copy-and-stop solutions while fp32 reasons. The broader sample below shows that was
> wrong: **bf16@100k reasons too** — long chain-of-thought with *every arithmetic step correct
> mod 23*. bf16 and fp32 behave nearly identically. Precision is **not** the differentiator.

Side-by-side greedy generations on the **same** Figure-3 (med) problems, fp32 inference on CPU
(the exact `src.eval.generate.IgsmGenerator` path: greedy, `EOS=50256`, no sampling).

**The real finding — the failure mechanism is over-generation, shared by both models:**

- Both models produce a well-formed chain-of-thought in which **every `N op M = R` clause is
  arithmetically correct mod 23** (no precision noise).
- Both models then **fail to stop**: they keep computing *extra* variables beyond the one the query
  asks for, and emit the **last-computed** value as the answer. Since the queried variable was
  usually computed earlier, the emitted answer is systematically wrong — which is why Figure-3 OOD
  accuracy is ~0 even though the math is perfect.
- This is the *planning / shortest-solution* skill the paper highlights ("the model can learn to
  generate shortest solutions, almost always avoiding unnecessary computations"). Neither model here
  has learned it — bf16@100k *after full training* still over-generates, and fp32 (~8 % trained)
  behaves the same way.

**Implication.** Because bf16@100k (fully trained) and fp32 behave identically on generation,
**precision alone will not fix the OOD failure.** The risk is real that fp32@100k also over-generates
and fails OOD. The thing to watch across milestones is whether fp32's solutions get *shorter* and
stop at the queried variable — not eval_loss (which is maxed at the irreducible entropy floor; see
`eval_loss_floor_irreducible.ipynb`) and not raw arithmetic correctness (already perfect).

> Data captured by `notebooks/_gen_examples.py` (CPU, ~10 min, does not touch the GPU training).
> Re-run it to refresh with newer fp32 checkpoints.

In [1]:
import json, os, re
d = os.getcwd()
while not os.path.exists(os.path.join(d, 'pyproject.toml')) and d != '/':
    d = os.path.dirname(d)
os.chdir(d)

DATA = json.load(open('notebooks/bf16_vs_fp32_examples.json'))
print('bf16 model :', DATA['bf16_model'])
print('fp32 model :', DATA['fp32_model'], ' (latest kept checkpoint)')
print('cap        : max_new_tokens =', DATA['max_new_tokens'])
print('slices     :', list(DATA['slices'].keys()))

bf16 model : models/100k_model/gpt-rope-igsm-fixed
fp32 model : models/gpt2-rope-igsm-100k-fp32/checkpoint-8000  (latest kept checkpoint)
cap        : max_new_tokens = 512
slices     : ['med_pq_op_eq15', 'med_pq_op_eq20', 'med_pq_op_eq23']


In [2]:
def arith_block(rows):
    if not rows:
        return '  (no  N op M = R  clauses -- model did no arithmetic)'
    return '\n'.join(
        f"  {r['expr']:18s}  correct mod23 = {r['correct_mod23']:>3}  {'OK' if r['ok'] else '*** WRONG ***'}"
        for r in rows
    )

def final_answer(text):
    """Last bare number the model emits (the value it stops on / submits as the answer)."""
    nums = re.findall(r'\b(\d+)\b', text)
    return nums[-1] if nums else None

def n_defines(text):
    return len(re.findall(r"\bDefine\b", text))

def show(entry):
    sl, i = entry['slice'], entry['idx']
    bf, fp = entry['bf16_100k'], entry['fp32']
    print('=' * 96)
    print(f"SLICE {sl}   problem #{i}   (prompt {entry['prompt_len']} tokens)")
    print('-' * 96)
    print('PROMPT (head):')
    print(entry['prompt_text'][:500].replace('\n', ' '))
    print('-' * 96)
    for name, m in [('bf16@100k', bf), ('fp32', fp)]:
        all_ok = all(r['ok'] for r in m['arithmetic'])
        print(f"--- {name}: {m['gen_len']} tokens, max pos {m['max_pos']}, "
              f"{n_defines(m['gen_text'])} 'Define' steps, "
              f"{len(m['arithmetic'])} arith clauses (all correct: {all_ok}), "
              f"final emitted value = {final_answer(m['gen_text'])} ---")
        print(m['gen_text'])
        print('   arithmetic check:')
        print('   ' + arith_block(m['arithmetic']).replace('\n', '\n   '))
        print()

## op_eq15 — hardest **in-distribution** bucket (training was op <= 15). bf16 scored 0.375 here.

In [3]:
for e in DATA['slices'].get('med_pq_op_eq15', []):
    show(e)

SLICE med_pq_op_eq15   problem #0   (prompt 413 tokens)
------------------------------------------------------------------------------------------------
PROMPT (head):
<|endoftext|>� The number of each Milk's Rosemary equals 5 more than the sum of each Brie Cheese's Rosemary, each Brie Cheese's Ingredient and each Brie Cheese's Nutmeg. The number of each Albertsons's Blue Cheese equals 8. The number of each Walmart's Sour Cream equals the difference of each Sour Cream's Cayenne Pepper and each Sour Cream's Nutmeg. The number of each Walmart's Blue Cheese equals 3. The number of each Walmart's Brie Cheese equals each Milk's Cayenne Pepper. The number of each Br
------------------------------------------------------------------------------------------------
--- bf16@100k: 248 tokens, max pos 661, 8 'Define' steps, 12 arith clauses (all correct: True), final emitted value = 17 ---
 Define Brie Cheese's Rosemary as K; so K = 8. Define Brie Cheese's Cayenne Pepper as T; so T = 17 * K = 17 *

## op_eq20 — OOD. bf16 scored 0.0.

In [4]:
for e in DATA['slices'].get('med_pq_op_eq20', []):
    show(e)

SLICE med_pq_op_eq20   problem #0   (prompt 295 tokens)
------------------------------------------------------------------------------------------------
PROMPT (head):
<|endoftext|>� The number of each Pine Ridge Elementary's Visual Arts Studio equals 13 more than the sum of each Pine Ridge Elementary's Theater Arts Classroom, each Sunnydale Primary's Theater Arts Classroom and each Pine Ridge Elementary's Dance Studio. The number of each Pine Ridge Elementary's Dance Studio equals each Sunnydale Primary's Theater Arts Classroom. The number of each Sunnydale Primary's Dance Studio equals the sum of each Sunnydale Primary's Visual Arts Studio, each Lakeview Ele
------------------------------------------------------------------------------------------------
--- bf16@100k: 259 tokens, max pos 554, 8 'Define' steps, 12 arith clauses (all correct: True), final emitted value = 19 ---
 Define Sunnydale Primary's Theater Arts Classroom as Y; so Y = 12. Define Pine Ridge Elementary's Dance Stud

## op_eq23 — OOD (longest). bf16 scored 0.0.

In [5]:
for e in DATA['slices'].get('med_pq_op_eq23', []):
    show(e)

SLICE med_pq_op_eq23   problem #0   (prompt 310 tokens)
------------------------------------------------------------------------------------------------
PROMPT (head):
<|endoftext|>� The number of each Small Mammal House's Pacman Frog equals the sum of each Mikumi National Park's Tiger Territory and each Camel Yard's Cave Salamander. The number of each Camel Yard's Cave Salamander equals each Mikumi National Park's Tiger Territory. The number of each Etosha National Park's Tiger Territory equals 19 more than each Etosha National Park's Camel Yard. The number of each Mikumi National Park's Small Mammal House equals 8 times as much as each Mikumi National Park's
------------------------------------------------------------------------------------------------
--- bf16@100k: 308 tokens, max pos 618, 13 'Define' steps, 9 arith clauses (all correct: True), final emitted value = 9 ---
 Define Mikumi National Park's Tiger Territory as f; so f = 14. Define Mikumi National Park's Camel Yard as s;

## Summary — bf16 and fp32 are nearly indistinguishable on generation behavior

Both emit long, arithmetically-perfect chains and stop on the *last* variable they computed.

In [6]:
import pandas as pd
rows = []
for sl, entries in DATA['slices'].items():
    for e in entries:
        bf, fp = e['bf16_100k'], e['fp32']
        rows.append({
            'slice': sl,
            'bf16_gen_len': bf['gen_len'], 'fp32_gen_len': fp['gen_len'],
            'bf16_define_steps': n_defines(bf['gen_text']), 'fp32_define_steps': n_defines(fp['gen_text']),
            'bf16_arith_correct': all(r['ok'] for r in bf['arithmetic']),
            'fp32_arith_correct': all(r['ok'] for r in fp['arithmetic']),
            'bf16_final': final_answer(bf['gen_text']), 'fp32_final': final_answer(fp['gen_text']),
            'bf16_max_pos': bf['max_pos'], 'fp32_max_pos': fp['max_pos'],
        })
df = pd.DataFrame(rows)
df

,slice,bf16_gen_len,fp32_gen_len,bf16_define_steps,fp32_define_steps,bf16_arith_correct,fp32_arith_correct,bf16_final,fp32_final,bf16_max_pos,fp32_max_pos
0,med_pq_op_eq15,248,260,8,8,True,True,17,0,661,673
1,med_pq_op_eq15,299,318,12,13,True,True,12,12,537,556
2,med_pq_op_eq20,259,245,8,7,True,True,19,12,554,540
3,med_pq_op_eq20,284,254,13,11,True,True,18,19,594,564
4,med_pq_op_eq23,308,303,13,12,True,True,9,11,618,613
5,med_pq_op_eq23,304,303,14,13,True,True,13,22,665,664


## Reading this + what to actually do

| signal | bf16@100k | fp32 (~8 % trained) | conclusion |
|---|---|---|---|
| arithmetic correctness | all correct mod 23 | all correct mod 23 | precision is **not** the problem |
| chain length | ~250-310 tokens | ~245-318 tokens | nearly identical |
| failure mode | over-generates, emits last variable | same | shared mechanism |
| OOD accuracy | 0.0 | wrong on samples | both fail |

**So:** the bf16-vs-fp32 precision story was a red herring (born from one misleading sample).
The OOD failure is a **planning / over-generation** problem: the model computes extra variables and
submits the last one instead of the queried one. bf16@100k has it *after full training*; fp32 behaves
the same at 8 %.

**What this means for the live fp32 run:** do not expect fp32@100k to magically clear OOD just
because it is fp32 — it must *learn to stop*, and that is not guaranteed. Track across milestones:

1. **solution length** on op_eq15/op_eq20 (should *shrink* toward the minimal chain as planning
   matures) — the single most diagnostic signal,
2. **generation accuracy** on op_eq15/op_eq20 (the paper's actual metric),
3. **NOT** eval_loss (maxed at the entropy floor) and **NOT** arithmetic correctness (already perfect).

If fp32's solution length is not shrinking and accuracy is still ~0 by ~25-50k, the repro has a
deeper planning/curriculum/data issue worth investigating independent of precision.